# Notebook F2a: is the weak POPE count effect specific to one wording, and does it depend on the learned prior of the question? (GPU, about 35 min)

Notebook F gave +0.09 logits for $k{=}8$ vs $k{=}2$ with one sentence. A reviewer would ask: *is that just a weak wording?* and *does the effect depend on how strong the learned prior is for that particular question, not only for the split?* This notebook tests both on ~200 POPE questions (66 per split, half present, half absent).

**1. Wordings** (each at $k{=}2$ and $k{=}8$ yes out of 10, compared with no history and the neutral sentence):
- `sentence`: "In the last 10 questions the answer was yes $k$ times and no $10-k$ times." (as in F)
- `list`: "Answers to the last 10 questions: yes, yes, no, ..." (explicit sequence)
- `hint`: the sentence plus "Most of the time the answer is yes/no." (an explicit cue, an upper bound on what a prompt can do)
- `rate`: "In this dataset, 80% (or 20%) of the answers are yes." (a stated base rate)
- `after`: the sentence placed *after* the question, right before the answer (recency)

**2. The learned prior per question.** We run every question on a **blank gray image**: the answer then reflects what LLaVA *expects* for that object question without any evidence (the learned prior, $P_i$). We then ask: does $P_i$ predict the image-grounded answer on absent objects (a validity check of the proxy)? Does the count effect depend on $P_i$ question by question (a finer interaction test than the three splits)? And what is the count effect when there is *no* evidence (blank image) vs with evidence?

**Reading.** If every wording gives a small effect, the weak POPE count effect is not a wording artefact. If one wording (e.g. `hint`) gives a large effect, then POPE can be moved by a prompt, but a count of past answers is not what moves it. A slope of the count effect on $P_i$ would be the first sign that the in-context prior and the learned prior interact.

**Setup:** GPU on (2x T4), **Internet on**, Kaggle secret `KAGGLE_KEY` on, *Save & Run All (Commit)*. Checkpoints: `ponishbhatia3/llava-pve-f2a-ckpt` (resume on re-run). `QUICK = True` is a smoke test (~5 min). Tested on a mock model; the model/dataset loading is the same as in notebook F, which ran.

In [ ]:
# ===================== CONFIG =====================
MOCK = False
QUICK = False                # True = tiny smoke test (~5 min)
SEED = 3                     # a different question sample than notebook F
OUT = "/kaggle/working/F2a_out"
N_PER_SPLIT = 66                             # questions per POPE split (half present / half absent); 198 in total
NEUTRAL = "The picture was taken with a plain camera in a quiet room on a sunny afternoon."
if QUICK: N_PER_SPLIT = 12
import os, re, glob, json, math, time, zipfile, shutil, warnings, subprocess, sys
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
def say(s=""): print(s); SUMMARY.append(s)

In [ ]:
# ===================== CHECKPOINTS TO A KAGGLE DATASET (so a dead session loses nothing) =====================
KAGGLE_USER = "ponishbhatia3"
CKPT_SLUG = f"{KAGGLE_USER}/llava-pve-f2a-ckpt"          # private dataset, created on the first push, new version afterwards
PUSH = not MOCK                                         # set False to disable (then only /kaggle/working is saved)
PUSH_EVERY_MIN = 10
CKPT_DIR = "/kaggle/working/F2a_ckpt"
RUN_CFG = dict(SEED=SEED, QUICK=QUICK, N_PER_SPLIT=N_PER_SPLIT)
json.dump(RUN_CFG, open(f"{OUT}/run_config.json", "w"))
_last_push = 0.0

def _setup_creds():
    assert KAGGLE_USER and "YOUR" not in KAGGLE_USER.upper(), "KAGGLE_USER is a placeholder"
    from kaggle_secrets import UserSecretsClient
    sc, key = UserSecretsClient(), None
    for name in ("KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        try: key = sc.get_secret(name); break
        except Exception: pass
    assert key, "No Kaggle secret found. Add-ons -> Secrets -> add KAGGLE_KEY (your Kaggle API key / token from Settings -> API), switch it ON for this notebook, rerun. Or set PUSH = False."
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_API_TOKEN" if key.startswith("KGAT_") else "KAGGLE_KEY"] = key

def _api():
    from kaggle.api.kaggle_api_extended import KaggleApi
    a = KaggleApi(); a.authenticate(); return a

def push_ckpt(note):
    os.makedirs(CKPT_DIR, exist_ok=True)
    open(f"{CKPT_DIR}/README.txt", "w").write("Checkpoints of notebook F2a (POPE wordings and learned prior). Resume by attaching this dataset as input and rerunning with the same settings.\n")
    for pat in ("*.csv", "*.png", "*.txt", "*.json"):
        for f in glob.glob(f"{OUT}/{pat}"): shutil.copy(f, CKPT_DIR)
    json.dump({"title": "LLaVA PVE F2a checkpoints", "id": CKPT_SLUG, "licenses": [{"name": "CC0-1.0"}]}, open(f"{CKPT_DIR}/dataset-metadata.json", "w"))
    api = _api()
    try: api.dataset_status(CKPT_SLUG); exists = True
    except Exception: exists = False
    res = api.dataset_create_version(CKPT_DIR, note, quiet=True, dir_mode="zip") if exists else api.dataset_create_new(CKPT_DIR, public=False, quiet=True, dir_mode="zip")
    err = getattr(res, "error", None)
    assert not err, f"Kaggle returned an error: {err}"
    print(f"[checkpoint] {'new version' if exists else 'created dataset'} {CKPT_SLUG} ({note}) | server says: {getattr(res, 'status', res)}", flush=True)

def maybe_push(note, force=False):
    global _last_push
    if not PUSH or (not force and time.time() - _last_push < PUSH_EVERY_MIN * 60): return
    try: push_ckpt(note); _last_push = time.time()
    except Exception as e: print(f"!!! CHECKPOINT PUSH FAILED (the run continues; files are still in {OUT}): {type(e).__name__}: {str(e)[:300]}", flush=True)

# resume: if a previous checkpoint dataset is attached as input with the SAME settings, copy its files back
for hit in glob.glob("/kaggle/input/**/run_config.json", recursive=True):
    try:
        if "llava-pve-outputs-run" in hit or json.load(open(hit)) != json.loads(json.dumps(RUN_CFG)): continue
    except Exception: continue
    d = os.path.dirname(hit)
    for name in ("wording_rows.csv",):
        if os.path.exists(f"{d}/{name}") and not os.path.exists(f"{OUT}/{name}"): shutil.copy(f"{d}/{name}", f"{OUT}/{name}"); print("resumed", name, "from", d)

if PUSH:                                                # fail fast: a real test push now, not after 40 minutes
    _setup_creds(); push_ckpt("preflight")
    print("checkpoint push works; later pushes happen every", PUSH_EVERY_MIN, "min and at the end of each stage")

In [ ]:
# ===================== POPE questions (stratified sample) =====================
class FakeImg:                                   # mock only
    def __init__(s, **kw): s.__dict__.update(kw)
def load_pope():
    if MOCK:
        rows = []
        for sp in ("random", "popular", "adversarial"):
            for i in range(N_PER_SPLIT): rows.append(dict(qid=f"{sp}{i}", split=sp, truth=int(i % 2), question=f"Is there a thing{i % 7} in the image?", img_id=f"img{i % 40}"))
        df = pd.DataFrame(rows); df["image"] = [FakeImg(truth=t, split=s, i=k, blank=False) for k, (t, s) in enumerate(zip(df.truth, df.split))]; return df
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset("lmms-lab/POPE")
    if hasattr(ds, "keys"): ds = concatenate_datasets([ds[k] for k in ds.keys()])
    cols = ds.column_names; print("POPE columns:", cols, "| rows:", len(ds))
    def pick(*names):
        for n in names:
            if n in cols: return n
        raise AssertionError(f"POPE loader: none of {names} found in columns {cols}. Send me this message.")
    cq, ca, cc, ci = pick("question"), pick("answer", "label"), pick("category", "split", "type"), pick("image")
    cid = next((n for n in ("image_source", "image_id", "image_name") if n in cols), None)
    meta = ds.remove_columns([ci]).to_pandas(); meta["_row"] = np.arange(len(meta))
    meta["truth"] = meta[ca].astype(str).str.lower().str.strip().map({"yes": 1, "no": 0}); assert meta.truth.notna().all()
    meta["split"] = meta[cc].astype(str).str.lower(); meta["img_id"] = meta[cid].astype(str) if cid else meta["_row"].astype(str)
    meta["question"] = meta[cq]; meta["qid"] = meta["_row"].astype(str)
    rng = np.random.default_rng(SEED); keep = []
    for sp in ("random", "popular", "adversarial"):
        for t in (1, 0):
            pool = meta[(meta.split == sp) & (meta.truth == t)]._row.values; assert len(pool) >= N_PER_SPLIT // 2
            keep += list(rng.choice(pool, N_PER_SPLIT // 2, replace=False))
    sub = meta[meta._row.isin(keep)].copy(); sub["image"] = [ds[int(r)][ci].convert("RGB") for r in sub._row]
    return sub[["qid", "split", "truth", "question", "img_id", "image"]].reset_index(drop=True)
Q = load_pope(); assert len(Q) == 3 * N_PER_SPLIT; print(Q.groupby(["split", "truth"]).size().unstack()); print("images:", Q.img_id.nunique())
_BL = {}
def blank_img(i):
    if MOCK: return FakeImg(truth=0, split=Q.split.iloc[i], i=i, blank=True)
    if "g" not in _BL: _BL["g"] = Image.new("RGB", (336, 336), (128, 128, 128))
    return _BL["g"]

In [ ]:
# ===================== wordings =====================
SEQ = {2: "no, no, yes, no, no, no, yes, no, no, no", 8: "yes, yes, no, yes, yes, yes, no, yes, yes, yes"}
SENT = lambda k: f"In the last 10 questions the answer was yes {k} times and no {10 - k} times."
FORMS = {"sentence": SENT, "list": lambda k: f"Answers to the last 10 questions: {SEQ[k]}.",
         "hint": lambda k: SENT(k) + f" Most of the time the answer is {'yes' if k > 5 else 'no'}.", "rate": lambda k: f"In this dataset, {k * 10}% of the answers are yes.", "after": SENT}
def build(form, k, q):
    core = f"{q}\nAnswer the question using a single word or phrase."
    if form is None: return f"USER: <image>\n{core} ASSISTANT:"
    if form == "neutral": return f"USER: <image>\n{NEUTRAL}\n{core} ASSISTANT:"
    if form == "after": return f"USER: <image>\n{core}\n{FORMS['after'](k)} ASSISTANT:"
    return f"USER: <image>\n{FORMS[form](k)}\n{core} ASSISTANT:"
# (name, image kind, form, k)
VARS = [("none", "real", None, 0), ("neutral", "real", "neutral", 0)] + [(f"{f}_k{k}", "real", f, k) for f in ("sentence", "list", "hint", "rate", "after") for k in (2, 8)] + [("blank_none", "blank", None, 0), ("blank_k2", "blank", "sentence", 2), ("blank_k8", "blank", "sentence", 8)]
if MOCK:
    class Runner:
        def prep(s, img, text):
            k = None
            for pat in (r"yes (\d+) times", r"(\d+)% of the answers"):
                m = re.search(pat, text)
                if m: k = int(m.group(1)) // (10 if "%" in pat else 1)
            if "Answers to the last 10 questions:" in text: k = 8 if "yes, yes" in text else 2
            return dict(img=img, k=k, strength=2.0 if "Most of the time" in text else 1.0, neutral=NEUTRAL in text)
        def delta(s, inp):
            im = inp["img"]; rr = np.random.default_rng(im.i); prior = rr.normal()
            ev = 0.0 if im.blank else (2 * im.truth - 1) * {"random": 3.0, "popular": 2.2, "adversarial": 1.4}[im.split]
            d = ev + 1.0 * prior - 1.0 + rr.normal(0, 0.3)
            if inp["k"] is not None: d = 0.5 * d + 0.12 * (inp["k"] - 5) * 0.6 * inp["strength"] * (1 + 0.3 * np.tanh(prior))
            return float(d)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=False)
    import torch
    from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
    NAME = "llava-hf/llava-1.5-7b-hf"
    model = LlavaForConditionalGeneration.from_pretrained(NAME, torch_dtype=torch.float16, device_map="auto", quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)).eval()
    proc = AutoProcessor.from_pretrained(NAME)
    NO_ID, YES_ID = [proc.tokenizer.encode(w, add_special_tokens=False)[0] for w in ("No", "Yes")]
    print("token ids No/Yes:", NO_ID, YES_ID)
    class Runner:
        def prep(s, img, text): return proc(images=img, text=text, return_tensors="pt").to(model.device, torch.float16)
        def delta(s, inp):
            with torch.no_grad(): lg = model(**inp).logits[0, -1].float()
            return float((lg[YES_ID] - lg[NO_ID]).item())
RUN = Runner()
for nm, kind, f, k in VARS[:1] + VARS[2:3] + VARS[4:5] + VARS[6:7] + VARS[8:9] + VARS[10:11]: print(f"[{nm}] " + build(f, k, "Is there a bed in the image?").replace("\n", " | "))
t0 = Q.iloc[0]; d_ = RUN.delta(RUN.prep(t0.image, build(None, 0, t0.question))); say(f"[self-test] first question '{t0.question}' truth={t0.truth}: delta(Yes-No) = {d_:+.2f}")
if not MOCK:
    top = proc.tokenizer.convert_ids_to_tokens(int(model(**RUN.prep(t0.image, build(None, 0, t0.question))).logits[0, -1].argmax())); say(f"[self-test] first token {top!r}"); assert top.replace("▁", "").strip().lower() in ("yes", "no")

In [ ]:
# ===================== run (checkpointed): 15 prompts per question =====================
pp = f"{OUT}/wording_rows.csv"; done = pd.read_csv(pp) if os.path.exists(pp) else pd.DataFrame()
have = set(done.qid.astype(str)) if len(done) else set(); rows = done.to_dict("records") if len(done) else []
todo = [i for i in range(len(Q)) if str(Q.qid.iloc[i]) not in have]; t0 = time.time(); print(f"questions: {len(Q)} x {len(VARS)} prompts; left: {len(todo)}")
for n_, i in enumerate(todo):
    r = Q.iloc[i]
    for nm, kind, f, k in VARS:
        img = r.image if kind == "real" else blank_img(i)
        rows.append(dict(qid=str(r.qid), split=r.split, truth=int(r.truth), img_id=r.img_id, variant=nm, delta=RUN.delta(RUN.prep(img, build(f, k, r.question)))))
    if (n_ + 1) % 20 == 0 or n_ == len(todo) - 1:
        pd.DataFrame(rows).to_csv(pp, index=False); maybe_push("wording"); el = time.time() - t0
        print(f"  {n_ + 1}/{len(todo)} | {el / 60:.1f} min | ~{el / (n_ + 1) * (len(todo) - n_ - 1) / 60:.0f} min left", flush=True)
R = pd.DataFrame(rows); R.to_csv(pp, index=False); maybe_push("wording done", force=True)

In [ ]:
# ===================== analysis 1: count effect, evidence cost and hallucination rate per wording =====================
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
W = R.pivot_table(index=["qid", "split", "truth", "img_id"], columns="variant", values="delta").reset_index(); SPL = ["random", "popular", "adversarial"]
FM = ["sentence", "list", "hint", "rate", "after"]; pres, ab = W.truth.values == 1, W.truth.values == 0
imgs = W.img_id.values; uimg = np.unique(imgs); idx_by = {u: np.where(imgs == u)[0] for u in uimg}; rb = np.random.default_rng(0)
def boot(stat, n=400):
    out = []
    for _ in range(n): pick = rb.choice(uimg, len(uimg)); ii = np.concatenate([idx_by[u] for u in pick]); out.append(stat(W.iloc[ii]))
    return np.array(out)
def ci(d, j): return np.percentile(d[:, j], [2.5, 97.5])
say(f"{len(W)} questions. Baseline (no history): AUROC {roc_auc_score(W.truth, W.none):.3f}; hit rate {(W[W.truth == 1].none > 0).mean():.3f}; false-yes {(W[W.truth == 0].none > 0).mean():.3f}; adversarial false-yes {(W[(W.truth == 0) & (W.split == 'adversarial')].none > 0).mean():.3f}")
def stat1(x):
    o = []
    for f in FM:
        g = x[f"{f}_k8"] - x[f"{f}_k2"]; sw0 = x[x.truth == 1].none.mean() - x[x.truth == 0].none.mean()
        o += [g.mean(), g[x.truth == 0].mean(), g[x.truth == 1].mean(), ((x[x.truth == 1][f"{f}_k8"].mean() - x[x.truth == 0][f"{f}_k8"].mean()) + (x[x.truth == 1][f"{f}_k2"].mean() - x[x.truth == 0][f"{f}_k2"].mean())) / 2 / sw0, (x[f"{f}_k2"] + x[f"{f}_k8"]).mean() / 2 - x.none.mean()]
    return np.array(o)
a0, ad = stat1(W), boot(stat1); rows = []
for j, f in enumerate(FM):
    b = 5 * j; rows.append(dict(wording=f, gap_all=a0[b], lo=ci(ad, b)[0], hi=ci(ad, b)[1], gap_absent=a0[b + 1], gap_present=a0[b + 2], swing_ratio=a0[b + 3], presence_shift=a0[b + 4]))
A1 = pd.DataFrame(rows); A1.to_csv(f"{OUT}/a1_wordings.csv", index=False)
say("\n[A1] per wording: gap = mean[Delta(k=8 yes of 10) - Delta(k=2)] in logits (gratings: +0.66; F's sentence on 510 questions: +0.09), with image-clustered 95% CI; swing_ratio = evidence swing relative to no history (gratings 0.4-0.55); presence_shift = mean over k=2,8 of Delta minus Delta(no history): what merely HAVING the sentence does")
say(A1.round(3).to_string(index=False)); say(f"   neutral sentence: presence shift {(W.neutral - W.none).mean():+.3f}, swing ratio {(W[W.truth == 1].neutral.mean() - W[W.truth == 0].neutral.mean()) / (W[W.truth == 1].none.mean() - W[W.truth == 0].none.mean()):.2f}")
say("\n[A1b] hallucination and hit rates by wording and count (false-yes = share of ABSENT objects answered yes; hit = share of PRESENT objects answered yes), all splits / adversarial only")
rows = []
for v in ["none", "neutral"] + [f"{f}_k{k}" for f in FM for k in (2, 8)]:
    rows.append(dict(prompt=v, false_yes=(W[ab][v] > 0).mean(), hit=(W[pres][v] > 0).mean(), false_yes_adv=(W[ab & (W.split == "adversarial").values][v] > 0).mean()))
say(pd.DataFrame(rows).round(3).to_string(index=False))
def stat2(x):
    o = []
    for f in FM:
        for sp in SPL: y = x[x.split == sp]; g = y[f"{f}_k8"] - y[f"{f}_k2"]; o += [g[y.truth == 0].mean(), g[y.truth == 1].mean()]
    return np.array(o)
s0, sdd = stat2(W), boot(stat2); rows = []
for a_, f in enumerate(FM):
    for b_, sp in enumerate(SPL): j = a_ * 6 + b_ * 2; rows.append(dict(wording=f, split=sp, gap_absent=s0[j], lo_a=ci(sdd, j)[0], hi_a=ci(sdd, j)[1], gap_present=s0[j + 1], lo_p=ci(sdd, j + 1)[0], hi_p=ci(sdd, j + 1)[1]))
say("\n[A1c] count effect by split for each wording (is there ever a split difference?)"); say(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
# ===================== analysis 2: the learned prior per question (blank-image answers) =====================
P = W.blank_none.values; E = W.none.values; gap = (W.sentence_k8 - W.sentence_k2).values
say(f"\n[A2] learned-prior proxy P_i = delta on a BLANK gray image (mean {P.mean():+.2f}, sd {P.std():.2f}, range {P.min():+.2f} to {P.max():+.2f})")
rho_a, rho_p = spearmanr(P[ab], E[ab])[0], spearmanr(P[pres], E[pres])[0]
say(f"   validity: Spearman(P_i, real-image delta) on ABSENT objects {rho_a:+.2f}, on PRESENT objects {rho_p:+.2f}; mean P_i by split: " + ", ".join(f"{sp} mean P {P[(W.split == sp).values].mean():+.2f}" for sp in SPL))
say(f"   among ABSENT objects: AUROC of P_i for 'the model said yes with the real image' = " + (f"{roc_auc_score((E[ab] > 0).astype(int), P[ab]):.2f}" if 0 < (E[ab] > 0).sum() < ab.sum() else "n/a (no or all yes)"))
def slopes(x):
    p, e, g = x.blank_none.values, x.none.values, (x.sentence_k8 - x.sentence_k2).values
    Z = np.c_[np.ones(len(p)), (p - p.mean()) / p.std(), (e - e.mean()) / e.std()]; b = np.linalg.lstsq(Z, g, rcond=None)[0]; b1 = np.linalg.lstsq(Z[:, :2], g, rcond=None)[0]
    return np.array([b[1], b[2], b1[1], np.corrcoef(g, p)[0, 1]])
s0, sd2 = slopes(W), boot(slopes)
say("count effect (k=8 minus k=2, sentence wording) regressed on standardized P_i and E_i = real-image delta without history (slope per 1 sd, image-clustered 95% CI):")
for j, nm in enumerate(["slope on P_i (controlling for E_i)", "slope on E_i (controlling for P_i)", "slope on P_i alone", "Pearson r(gap, P_i)"]): say(f"   {nm:36s} {s0[j]:+.3f} [{ci(sd2, j)[0]:+.3f}, {ci(sd2, j)[1]:+.3f}]")
W["Ptert"] = pd.qcut(W.blank_none, 3, labels=["low P (model expects 'no')", "mid P", "high P (model expects 'yes')"])
say("count effect by tertile of P_i:"); say(W.groupby("Ptert", observed=True).apply(lambda y: pd.Series(dict(n=len(y), gap=(y.sentence_k8 - y.sentence_k2).mean(), false_yes_none=(y[y.truth == 0].none > 0).mean(), P_mean=y.blank_none.mean()))).round(3).to_string())
gb = W.blank_k8 - W.blank_k2; sw_real = [W[W.truth == t].sentence_k8.mean() - W[W.truth == t].sentence_k2.mean() for t in (1, 0)]
say(f"\n[A2b] count effect when there is NO evidence (blank image): {gb.mean():+.3f} (real images: {gap.mean():+.3f}); per-question Spearman(blank effect, real effect) = {spearmanr(gb, gap)[0]:+.2f}")
def rule(dn, dk2, dk8):
    X = np.c_[np.r_[dn, dn], np.r_[-np.ones(len(dn)), np.ones(len(dn))], np.ones(2 * len(dn))]; return np.linalg.lstsq(X, np.r_[dk2, dk8], rcond=None)[0]
for f in FM: b = rule(W.none.values, W[f"{f}_k2"].values, W[f"{f}_k8"].values); say(f"   gain/offset rule on real images, {f:9s}: gain a = {b[0]:.2f}, offset c = {b[1]:+.3f}   (gratings: a 0.43, c 0.63)")
fig, ax = plt.subplots(1, 3, figsize=(16, 4)); ax[0].bar(range(len(FM)), A1.gap_all, yerr=[A1.gap_all - A1.lo, A1.hi - A1.gap_all]); ax[0].set_xticks(range(len(FM))); ax[0].set_xticklabels(FM); ax[0].axhline(0.66, c="k", ls=":"); ax[0].set_title("count effect per wording (dotted: gratings +0.66)")
ax[1].scatter(P, gap, s=8, c=np.where(ab, "C1", "C0")); ax[1].set_xlabel("P_i (blank-image delta)"); ax[1].set_ylabel("count effect k8-k2"); ax[1].set_title("count effect vs learned-prior proxy")
ax[2].scatter(gb, gap, s=8); ax[2].set_xlabel("count effect, blank image"); ax[2].set_ylabel("count effect, real image"); ax[2].set_title("no evidence vs evidence"); plt.tight_layout(); plt.savefig(f"{OUT}/f2a_overview.png", dpi=130); plt.close()

In [ ]:
open(f"{OUT}/F2a_summary.txt", "w").write("\n".join(SUMMARY)); maybe_push("final", force=True)
with zipfile.ZipFile(f"{OUT}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUT): zf.write(f"{OUT}/{f}", f)
print(f"zip written: {OUT}.zip")